# Exploración del Dataset NoisyUAV v2

**Objetivo:** Visualización y comprensión de las señales RF I/Q del dataset para detección de drones.

**Dataset:** Glüge et al. (2024) — *Robust Low-Cost Drone Detection and Classification Using CNNs in Low SNR Environments*

| Parámetro | Valor |
|---|---|
| Fs (muestreo) | 14 MHz (downsampled de 56 MHz) |
| Muestras/archivo | 1,048,576 (2²⁰) |
| Duración | ~74.9 ms |
| Clases | 6 drones + 1 ruido = 7 |
| SNR | [-20, 30] dB, paso 2 dB |
| Total muestras | 17,744 |

In [1]:
import sys
import os

# Añadir la carpeta padre al path para poder importar 'funciones'
sys.path.insert(0, r"c:\repos\DroneDetectionRF")

from NoisyUAV.funciones.dataset.cargador import (
    cargar_muestra,
    cargar_metadatos_dataset,
    obtener_una_muestra,
    obtener_muestras_por_clase,
    NOMBRES_CLASES,
    FREQ_MUESTREO,
    DURACION_MUESTRA,
    DATA_DIR,
    TARGET_NOISE,
)
from NoisyUAV.funciones.visualizacion.visualizacion import (
    panel_completo,
    comparar_snr,
    comparar_clases,
)
# from NoisyUAV.funciones.cargador import obtener_una_muestra

import matplotlib.pyplot as plt
%matplotlib inline

print(f"Directorio de datos: {DATA_DIR}")
print(f"Frecuencia de muestreo: {FREQ_MUESTREO/1e6} MHz")
print(f"Duración por muestra: {DURACION_MUESTRA*1000:.1f} ms")
print(f"Clases: {NOMBRES_CLASES}")
print(f"Índice de clase Noise: {TARGET_NOISE}")

Directorio de datos: C:\TFM_data\NoisyUAV\drone_RF_data
Frecuencia de muestreo: 14.0 MHz
Duración por muestra: 74.9 ms
Clases: {0: 'DJI', 1: 'FutabaT14', 2: 'FutabaT7', 3: 'Graupner', 4: 'Noise', 5: 'Taranis', 6: 'Turnigy'}
Índice de clase Noise: 4


## 2. Metadatos del Dataset

In [ ]:
class_stats, snr_stats = cargar_metadatos_dataset()

print("=" * 50)
print("DISTRIBUCIÓN POR CLASE")
print("=" * 50)
display(class_stats)

print("\n")
print("=" * 50)
print("DISTRIBUCIÓN POR SNR")
print("=" * 50)
display(snr_stats)

## 3. Cargar y Examinar una Muestra

In [ ]:
# Elegimos un dron Taranis (target=5) a SNR alto para ver la señal limpia
rutas = obtener_muestras_por_clase(target=0, snr=-20, n=10)
rutas

In [ ]:
iq, sample_id, target, snr = cargar_muestra(rutas[0])

print(f"Archivo: {os.path.basename(rutas[0])}")
print(f"Shape del tensor: {iq.shape}")
print(f"Clase (target): {target} → {NOMBRES_CLASES[target]}")
print(f"SNR: {snr} dB")
print(f"Sample ID: {sample_id}")
print(f"Rango I: [{iq[0].min():.6f}, {iq[0].max():.6f}]")
print(f"Rango Q: [{iq[1].min():.6f}, {iq[1].max():.6f}]")
print(f"Tipo de dato: {iq.dtype}")

## 4. Panel Completo de Visualización

Visualización en 4 dominios de una señal de dron a **alto SNR** (señal limpia, fácil de interpretar).

In [ ]:
resultado_dron = obtener_una_muestra(target=4, snr=-16, index=6)
tensor_dron = resultado_dron[0]

In [ ]:
resultado_dron

In [ ]:
fig = panel_completo(tensor_dron, 4, -16)
plt.show()

In [ ]:
iq_noise, sid_noise, tgt_noise, snr_noise = obtener_una_muestra(target=TARGET_NOISE, snr=-10)
fig = panel_completo(iq_noise, tgt_noise, snr_noise, sample_id=sid_noise)
plt.show()

## 5. Comparación por SNR

¿Cómo se degrada la señal de un mismo dron a medida que baja el SNR?

Esto es clave para entender el **reto de detección a SNR negativos**.

In [ ]:
fig = comparar_snr(
    target=2,  # Taranis (índice 5 en orden alfabético)
    snr_list=[-20, -10, 0, 10, 20, 30],
)
plt.show()

## 6. Comparación por Clase

¿Cómo se ven las distintas firmas RF de cada dron al mismo SNR?

Cada dron tiene un patrón de **frequency hopping** y una **modulación** característica.

In [ ]:
fig = comparar_clases(
    snr=-12,
    targets=[0, 1, 2, 3, 4, 5, 6],  # Todas las clases (4=Noise)
)
plt.show()

In [2]:
# ===== GENERACIÓN Y GUARDADO AUTOMÁTICO DE LAS FIGURAS DEL TFM =====
import os

# Carpeta de destino para las figuras del TFM
output_dir = r"c:\repos\DroneDetectionRF\NoisyUAV\figuras_TFM_reducidas"
os.makedirs(output_dir, exist_ok=True)

print(f"📁 Guardando figuras en: {output_dir}\n")

# 1. Ejemplos de Dataset (panel_completo) para Dron DJI (target=0) y Ruido (target=4)
# Niveles de SNR a generar: -16 dB, 0 dB, 16 dB
ejemplos = [
    (0, 30), (0, 12), (0, 0), (0, -12), (0, -20),  # Dron DJI
    (4, 30), (4, 12), (4, 0), (4, -12), (4, -20),  # Ruido
]

for tgt, snr_val in ejemplos:
    try:
        iq_tensor, sample_id, _, _ = obtener_una_muestra(target=tgt, snr=snr_val)
        fig = panel_completo(iq_tensor, target=tgt, snr=snr_val, sample_id=sample_id)
        filename = f"ej_dataset_target{tgt}_{snr_val}db.png"
        path = os.path.join(output_dir, filename)
        fig.savefig(path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"✅ Guardado: {filename}")
    except Exception as e:
        print(f"❌ Error al generar target={tgt}, snr={snr_val}: {e}")

# 2. Comparación por rango de SNR para FutabaT7 (target=2)
try:
    fig = comparar_snr(target=2, snr_list=[30, 12, 0, -12, -20])
    filename = "ej_rangoSNR_target2.png"
    path = os.path.join(output_dir, filename)
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"✅ Guardado: {filename}")
except Exception as e:
    print(f"❌ Error al generar comparación SNR target 2: {e}")

# 3. Comparación por clases para SNR = -12 dB, 0 dB, y 22 dB
snrs_clases = [-12, 0, 10, 28]
for snr_val in snrs_clases:
    try:
        fig = comparar_clases(snr=snr_val, targets=[0, 1, 2, 3, 4, 5, 6])
        filename = f"comparacion_clase_{snr_val}dB.png"
        path = os.path.join(output_dir, filename)
        fig.savefig(path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"✅ Guardado: {filename}")
    except Exception as e:
        print(f"❌ Error al generar comparación clases para SNR={snr_val}: {e}")

print("\n🎉 ¡Proceso finalizado! Todas las figuras han sido actualizadas.")

📁 Guardando figuras en: c:\repos\DroneDetectionRF\NoisyUAV\figuras_TFM_reducidas



c:\repos\DroneDetectionRF\NoisyUAV\funciones\visualizacion\visualizacion.py:245: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


✅ Guardado: ej_dataset_target0_30db.png


c:\repos\DroneDetectionRF\NoisyUAV\funciones\visualizacion\visualizacion.py:245: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


✅ Guardado: ej_dataset_target0_12db.png


...

✅ Guardado: ej_dataset_target4_0db.png


c:\repos\DroneDetectionRF\NoisyUAV\funciones\visualizacion\visualizacion.py:245: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


✅ Guardado: ej_dataset_target4_-12db.png


c:\repos\DroneDetectionRF\NoisyUAV\funciones\visualizacion\visualizacion.py:245: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


✅ Guardado: ej_dataset_target4_-20db.png
✅ Guardado: ej_rangoSNR_target2.png
✅ Guardado: comparacion_clase_-12dB.png
✅ Guardado: comparacion_clase_0dB.png
✅ Guardado: comparacion_clase_10dB.png
✅ Guardado: comparacion_clase_28dB.png

🎉 ¡Proceso finalizado! Todas las figuras han sido actualizadas.
